# ♻️ Triagem de Resíduos — Notebook corrigido para Colab

Este notebook contém as etapas de: conferir GPU, enviar dataset, (opcional) preparar TACO, treinar com Transfer Learning (MobileNetV2) e — importante — uma sequência segura para converter o modelo para TensorFlow.js alinhando versões (instala TF 2.19.0 + tensorflowjs e reiniciando o runtime).

INSTRUÇÕES IMPORTANTES:
1. Salve todo trabalho não salvo antes de executar a célula marcada 'FORÇAR REINÍCIO'. Ela reinstala TensorFlow e reinicia o runtime.
2. Após reconectar, execute as células na ordem: preparar/treinar (se desejar), depois a célula de conversão (após reinício).


## 1) Conferir GPU

In [ ]:
!nvidia-smi

## 2) Enviar o dataset (faça o upload do ZIP)

In [ ]:
from google.colab import files
import os, zipfile

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/upload_extracted')
print('Extraido em /content/upload_extracted')
!find /content/upload_extracted -maxdepth 4 -type d

In [ ]:
# Localiza automaticamente a pasta que contem as 6 subpastas de classes
import os
CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
candidate = None
for root, dirs, files_ in os.walk('/content/upload_extracted'):
    if all(c in dirs for c in CLASSES):
        candidate = root
        break
assert candidate is not None, "Nao encontrei uma pasta com as 6 subpastas esperadas. Confira o ZIP enviado."
DATASET_DIR = candidate
print("Dataset encontrado em:", DATASET_DIR)
for c in CLASSES:
    n = len(os.listdir(os.path.join(DATASET_DIR, c)))
    print(f"  {c}: {n} imagens")


## 3) (Opcional) Baixar e preparar o TACO
Se quiser mesclar o TACO, ajuste USE_TACO = True e rode as células.

In [ ]:
USE_TACO = False  #@param {type:"boolean"}
if USE_TACO:
    !git clone -q https://github.com/pedropro/TACO
    !pip install -q -r TACO/requirements.txt
    !python TACO/download.py
    print("TACO baixado.")
else:
    print("Pulando TACO (USE_TACO=False).")


## 4) Escreve o script de treino (Transfer Learning com MobileNetV2)

In [ ]:
%%writefile train_transfer_learning.py
import argparse
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

DATA_DIR = "Garbage classification"  # sobrescrito por --data_dir
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 30
CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]

def build_datasets():
    train_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR, validation_split=0.3, subset="training", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int",
        class_names=CLASSES,
    )
    val_test_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR, validation_split=0.3, subset="validation", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int",
        class_names=CLASSES,
    )
    val_batches = tf.data.experimental.cardinality(val_test_ds)
    test_ds = val_test_ds.take(val_batches // 2)
    val_ds = val_test_ds.skip(val_batches // 2)
    return train_ds, val_ds, test_ds

def build_augmentation():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.25),
        layers.RandomBrightness(0.25),
        layers.RandomTranslation(0.1, 0.1),
    ])

def build_model(num_classes, fine_tune_at=100):
    base = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
    base.trainable = True
    for layer in base.layers[:fine_tune_at]:
        layer.trainable = False

    augmentation = build_augmentation()
    inputs = layers.Input(shape=IMG_SIZE + (3,))
    x = augmentation(inputs)
    x = preprocess_input(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs, outputs)
    return model

def get_labels(ds):
    return np.concatenate([y.numpy() for _, y in ds])

def main():
    global DATA_DIR
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", default=DATA_DIR)
    parser.add_argument("--epochs", type=int, default=EPOCHS)
    parser.add_argument("--fine_tune_at", type=int, default=100)
    parser.add_argument("--out_model", default="modelo_residuos_mobilenetv2.keras")
    args = parser.parse_args()
    DATA_DIR = args.data_dir
    train_ds, val_ds, test_ds = build_datasets()
    y_train = get_labels(train_ds)
    class_weights = compute_class_weight(
        "balanced", classes=np.arange(len(CLASSES)), y=y_train
    )
    class_weight_dict = dict(enumerate(class_weights))
    train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
    test_ds = test_ds.prefetch(tf.data.AUTOTUNE)
    model = build_model(len(CLASSES), fine_tune_at=args.fine_tune_at)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5),
    ]
    history = model.fit(
        train_ds, validation_data=val_ds, epochs=args.epochs,
        class_weight=class_weight_dict, callbacks=callbacks,
    )
    # avaliação no teste
    y_true, y_pred = [], []
    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))
    cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))
    print("Matriz de confusão:")
    print(CLASSES)
    print(cm)
    print(classification_report(y_true, y_pred, target_names=CLASSES, zero_division=0))
    gi, mi = CLASSES.index("glass"), CLASSES.index("metal")
    print(f"Vidro->Metal: {cm[gi, mi]}  Metal->Vidro: {cm[mi, gi]}")
    model.save(args.out_model)
    print(f"Modelo salvo em {args.out_model}")
    # gráfico de treino
    import matplotlib.pyplot as plt
    plt.figure()
    plt.plot(history.history["accuracy"], label="treino")
    plt.plot(history.history["val_accuracy"], label="validação")
    plt.xlabel("Época")
    plt.ylabel("Acurácia")
    plt.legend()
    plt.title("Curva de treinamento")
    plt.savefig("curva_treinamento.png", dpi=150)

if __name__ == "__main__":
    main()


## 5) Rodar o treino (opcional)
Rode a célula abaixo para treinar (vai demorar dependendo do dataset).


In [ ]:
# Exemplo de execução do script de treino (descomente e ajuste se desejar)
# import subprocess
# cmd = ["python", "train_transfer_learning.py", "--data_dir", DATASET_DIR, "--epochs", "20", "--out_model", "/content/modelo_residuos_mobilenetv2.keras"]
# subprocess.run(cmd)
print('Se quiser treinar, descomente a chamada ao subprocess e rode esta célula.')


## 6) (CORRIGIDO) Forçar instalação compatível e reinício do runtime
Esta célula REINICIARÁ o runtime quando executada. Salve seu trabalho antes.


In [ ]:
# ATENÇÃO: esta célula reinstala TensorFlow 2.19.0 e tensorflowjs e REINICIA o runtime.
# Salve seu trabalho antes de executar.
import sys, subprocess, os, time

print("1) Removendo tensorflow_decision_forests (se estiver instalado)...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "tensorflow_decision_forests"], check=False)

print("2) Instalando TensorFlow 2.19.0 e tensorflowjs...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow==2.19.0", "tensorflowjs"], check=True)

print("Instalação concluída. Reiniciando o runtime para aplicar mudanças (o notebook será interrompido)...")
time.sleep(2)
os.kill(os.getpid(), 9)


## 7) (APÓS REINÍCIO) Converter o modelo Keras salvo para TensorFlow.js
Rode esta célula APÓS o reinício do runtime causado pela célula anterior.


In [ ]:
# Executar APÓS o reinício do runtime
import os
import tensorflow as tf
import tensorflowjs as tfjs

MODEL_KERAS_PATH = "/content/modelo_residuos_mobilenetv2.keras"
OUT_TFJS_DIR = "/content/modelo_tfjs"

print("TensorFlow versão detectada:", tf.__version__)
print("Verificando existência do modelo salvo em:", MODEL_KERAS_PATH)
if not os.path.exists(MODEL_KERAS_PATH):
    raise FileNotFoundError(f"Não encontrei o modelo em {MODEL_KERAS_PATH}. Verifique o caminho.")

print("Carregando o modelo Keras...")
model = tf.keras.models.load_model(MODEL_KERAS_PATH)
print("Modelo carregado. Convertendo para TF.js em:", OUT_TFJS_DIR)

# remove pasta de saída antiga se existir
if os.path.exists(OUT_TFJS_DIR):
    import shutil
    shutil.rmtree(OUT_TFJS_DIR)

tfjs.converters.save_keras_model(model, OUT_TFJS_DIR)
print("Conversão concluída. Conteúdo salvo em", OUT_TFJS_DIR)
print("Arquivos de saída (amostra):", os.listdir(OUT_TFJS_DIR)[:20])


## 8) (APÓS CONVERSÃO) Zipar e baixar os artefatos
Esta célula compacta a pasta TF.js e prepara o download do .keras e do zip TF.js.


In [ ]:
import os, shutil
from google.colab import files

KERAS_MODEL_PATH = "/content/modelo_residuos_mobilenetv2.keras"
TFJS_DIR = "/content/modelo_tfjs"
TFJS_ZIP = "/content/modelo_tfjs.zip"
KERAS_ZIP = "/content/modelo_residuos_mobilenetv2.keras.zip"

# zip TFJS se existir
if os.path.isdir(TFJS_DIR):
    if os.path.exists(TFJS_ZIP):
        os.remove(TFJS_ZIP)
    shutil.make_archive(TFJS_ZIP.replace('.zip',''), 'zip', TFJS_DIR)
    print("TF.js zip criado:", TFJS_ZIP)
else:
    print("Pasta TF.js não encontrada em", TFJS_DIR)

# zip do modelo Keras caso seja diretório
if os.path.exists(KERAS_MODEL_PATH):
    if os.path.isdir(KERAS_MODEL_PATH):
        if os.path.exists(KERAS_ZIP):
            os.remove(KERAS_ZIP)
        shutil.make_archive(KERAS_ZIP.replace('.zip',''), 'zip', KERAS_MODEL_PATH)
        to_download_keras = KERAS_ZIP
        print("Keras (diretório) zip criado:", KERAS_ZIP)
    else:
        to_download_keras = KERAS_MODEL_PATH
        print("Keras (arquivo) encontrado:", KERAS_MODEL_PATH)
else:
    to_download_keras = None
    print("Modelo Keras não encontrado em", KERAS_MODEL_PATH)

# iniciar downloads
if to_download_keras and os.path.exists(to_download_keras):
    print("Iniciando download:", to_download_keras)
    files.download(to_download_keras)
if os.path.exists(TFJS_ZIP):
    print("Iniciando download:", TFJS_ZIP)
    files.download(TFJS_ZIP)
